
# Task 2: Time-Series Regression (Remaining Useful Life/RUL)

## 1. Project Overview

### 1.1 Problem Definition

This project focuses on predicting the **Remaining Useful Life (RUL)** of press machines using time‑series sensor measurements collected throughout operational cycles. 

The primary objective is to estimate the number of remaining operating cycles before machine failure occurs, enabling valuable information for maintenance scheduling and reliability management. 

To improve model practicality and stability, the target variable is capped at a **maximum value of one hundred fifty operating cycles**.

$
RUL_t = \min(T_{\text{failure}} - t, 150)$

### 1.2 Project Objectives

1. Perform Exploratory Data Analysis (EDA).
2. Conduct data preprocessing and RUL label generation.
3. Transform time-series data into sliding windows.
4. Develop and compare sequence-based Deep Learning models.
5. Establish a non-Deep Learning baseline.
6. Perform machine-level validation.
7. Estimate prediction uncertainty.
8. Evaluate models using RMSE and Asymmetric Score.

### 1.3 Evaluation Metrics

- Root Mean Squared Error (RMSE)
- Asymmetric Score

### 1.4 Project Requirements

- At least two sequence-based Deep Learning architectures.
- One non-Deep Learning baseline.
- Machine-level train-validation split.
- Prediction intervals using Monte Carlo Dropout or Ensemble.
- Final RUL prediction for test machines.



## 2. Library Imports

Several Python libraries are imported to facilitate data processing, file operations, archive handling, and formatted output generation, forming the foundation for subsequent project implementations:

- **Pandas:** Used for data manipulation and tabular data analysis.
- **Pathlib:** Used to manage file and directory paths.
- **IPython.display:** Provides functions to display formatted outputs and render Markdown content within the notebook.
- **ZipFile:** Used to access and read files directly from a ZIP archive without manually extracting its contents.

Additional libraries will be imported in subsequent sections depending on the requirements of data analysis, preprocessing, visualization, and model development.


In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display, Markdown
from zipfile import ZipFile


## 3. Dataset Understanding

Before performing data analysis and model development, it is important to understand the structure and characteristics of the provided dataset.

This section provides an initial overview of the available dataset files, explains the meaning of relevant variables, and identifies the data sources required for the RUL regression task.

The dataset understanding process consists of the following steps:

1. Dataset inventory.
2. Dataset dictionary inspection.
3. RUL dataset variable interpretation.
4. Training and testing data identification.
5. Dataset mapping.


In [15]:
zip_path = Path(r"C:\Users\LENOVO\OneDrive\Documents\Telkom University\SEMESTER 9\Deep Learning\UTS\data_uts_dl.zip")

if zip_path.exists():
    with ZipFile(zip_path, "r") as zip_ref: #r means read, zip_ref is for the zip folder that opened
    files = [item for item in zip_ref.infolist() if not item.is_dir()] #infolist takes zip lists, dir represent if the item a file not a dictionary
    inventory = pd.DataFrame([{
    "No": idx + 1,
    "Nama File": file.filename,
    "Ukuran (MB)": file.file_size / (1024 * 1024) #byte to mb
} for idx, file in enumerate(files)])

else: print("Archive data_uts_dl.zip tidak ditemukan. Lanjut menggunakan folder data/.")

display(
    inventory.style
    .set_caption("Dataset Inventory")
    .set_table_styles([
        {
            "selector": "caption",
            "props": [
                ("text-align", "center"),
                ("font-size", "14px"),
                ("font-weight", "bold"),
                ("caption-side", "top"),
                ("padding-bottom", "8px")
            ]
        },
        {
            "selector": "th",
            "props": [("text-align", "center")]
        }
    ])
    .format({"Ukuran (MB)": "{:.2f}"})
    .set_properties(**{"text-align": "center"})
    .set_properties(subset=["Nama File"], **{"text-align": "left"})
    .hide(axis="index")
)

IndentationError: expected an indented block after 'with' statement on line 4 (2378138558.py, line 5)

In [ ]:
with ZipFile(zip_path, "r") as zip_ref:
    kamus_data = next( #next take the first file which fills the condition
        name for name in zip_ref.namelist() #take all file name on the folder
        if name.lower().endswith("kamus_data.md") #chose "kamus_data.md"
    )

    info = zip_ref.getinfo(kamus_data)
    kamus_text = zip_ref.read(kamus_data).decode("utf-8-sig", errors="replace") #read file then change it to UTF 8, change the unreadable character

display(Markdown(kamus_text))

# Kamus data UTS Deep Learning

Semua kelompok menerima data yang sama. Label data uji tidak dibagikan.

## Tugas 1: klasifikasi_train.npz, klasifikasi_test.npz, klasifikasi_*_meta.csv

Satu sampel per unit produk dari pabrik komponen fiktif. Baris ke-i di berkas meta
memakai `indeks` = i pada array di berkas npz.

| Isi | Keterangan |
| --- | --- |
| `gambar` | Array uint8 (N, 64, 64), foto permukaan produk |
| `sensor` | Array float16 (N, 3, 128): suhu (C), getaran (relatif), arus (A) selama penekanan. NaN berarti data tidak terekam |
| id_unit | ID unit produk |
| lini | Lini produksi (A atau B) |
| shift | Shift produksi |
| id_mesin | Mesin penekan |
| batch_material | Batch bahan baku |
| kelas | Label: normal, gores, noda, retak, gelembung, las_dingin, kontaminasi. Hanya di data latih |

Label data latih berasal dari inspektur dan tidak dijamin bebas salah.

## Tugas 2: rul_train.csv.gz dan rul_test.csv.gz

Log per siklus untuk mesin press fiktif. Data latih berisi riwayat sampai mesin rusak.
Data uji berhenti pada titik acak sebelum rusak.

| Kolom | Keterangan |
| --- | --- |
| id_unit | ID mesin |
| siklus | Nomor siklus sejak mesin mulai dipakai |
| setel_1, setel_2, setel_3 | Pengaturan operasi pada siklus itu |
| s01-s21 | Bacaan sensor. NaN berarti bacaan hilang |

Label: sisa umur pakai (RUL) = jumlah siklus tersisa sampai rusak, dipotong pada 150
(sisa umur di atas 150 siklus dihitung 150). Di data latih, siklus terakhir tiap unit adalah
siklus saat rusak, jadi labelnya dihitung sendiri. Prediksi RUL dibuat pada siklus terakhir
tiap unit di data uji.

## Tugas 3: clustering.npz dan clustering_meta.csv

Sampel material dari laboratorium fiktif, tanpa label.

| Isi | Keterangan |
| --- | --- |
| `gambar` | Array uint8 (N, 64, 64), citra mikrostruktur dari mikroskop |
| `spektrum` | Array float16 (N, 128), spektrum komposisi |
| id_sampel | ID sampel |
| mikroskop | Mikroskop yang dipakai (M1 atau M2) |
